# Boyajian's Star: century-long dimming, or the Menzel Gap?

The second step of [IDEAS.md idea 2](../IDEAS.md), and the reason the
[RY Cnc first-look](ry_cnc_century_of_plates.ipynb) came first: reproduce a *contested*
published result, where both sides are in the literature and the disagreement is entirely
about systematics.

**The dispute.** KIC 8462852 — Boyajian's Star — shows bizarre, deep, irregular day-scale dips
in Kepler data. In 2016, [Schaefer](https://iopscience.iop.org/article/10.3847/0004-637X/825/1/73)
measured its DASCH light curve and claimed a steady secular dimming of
**0.164 ± 0.013 mag/century** from 1890 to 1989.
[Hippke et al.](https://arxiv.org/abs/1601.07314) and
[Lund et al.](https://arxiv.org/abs/1605.02760) countered that comparison stars show
structure of the same size under the same reduction, and that the apparent trend is dominated
by a calibration discontinuity across the **Menzel Gap** (the 1953–1969 collapse of Harvard's
plate programme). The DASCH team disputed the rebuttal. *A decade later it is still
unresolved* — which makes it perfect: we are not trying to settle it, we are trying to earn an
opinion, with our own numbers, on why it is hard.

(Context that keeps the claim alive: [Montet & Simon](https://arxiv.org/abs/1608.01316) found
the star genuinely dimmed ~3% *within* the four-year Kepler mission, from space, beyond
dispute. So "this star secularly dims" is not inherently outlandish; the question is strictly
whether the *plate data* demonstrates it over a century.)

**The plan — three measurements, no new statistics:**

1. **Reproduce** — run our standard reduction on the DASCH DR7 light curve and fit a line.
   Do we get Schaefer's slope?
2. **Decompose** — fit the pre-gap era alone, and measure the offset across the gap. Is it a
   *trend*, or a *step* where the observatory changed everything?
3. **Control** — run the *identical* pipeline on field stars of matched brightness and color.
   The spread of their slopes is the honest error bar on ours.

In [ ]:
import os

for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(var, "4")

import numpy as np
import matplotlib.pyplot as plt

import daschlab

sess = daschlab.open_session("../data/cache/dasch/kic8462852")
sess.select_target(name="KIC 8462852")
sess.select_refcat("apass")

refcat = sess.refcat()
target = refcat[0]   # local_id 0 = nearest catalog source, 0.02" from the SIMBAD position
print(f"DASCH sees the target at B = {float(target['stdmag']):.2f}, color {float(target['color']):+.2f}")
print(f"(V = 11.9, B\u2212V \u2248 0.5 for an F3V star \u2014 consistent: DASCH plates are blue/B-band)")

**Source-splitting check** (the [known issue](https://dasch.cfa.harvard.edu/dr7/) that
notebook 1 flagged): the second-nearest catalog entry sits 13″ away with no calibrated
magnitude. Its light curve holds only ~24 detections against our ~2,700 — a stray fragment,
not a meaningful split, so we proceed with `lightcurve(0)` alone and skip merging.

## 1. Coverage first, always

Same drill as RY Cnc: before trusting any trend, look at *when* the plates exist. The Menzel
Gap will reappear in every plot below — every century-scale claim about this star has to step
across it.

In [ ]:
exposures = sess.exposures()

dated = ~exposures["obs_date"].mask        # undated plates would silently read as J2000
years_exp = exposures["obs_date"][dated].jyear

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.hist(years_exp, bins=np.arange(1885, 1995, 1), color="#39516b")
ax.axvspan(1953, 1969, color="orange", alpha=0.18)
ax.text(1961, ax.get_ylim()[1] * 0.75, "Menzel\nGap", ha="center", color="darkorange")
ax.set_xlabel("Year")
ax.set_ylabel("Exposures / year")
ax.set_title(f"{len(exposures):,} exposures cover KIC 8462852 ({years_exp.min():.0f}\u2013{years_exp.max():.0f})")
plt.show()

## 2. Measurement 1: reproduce the slope

Standard reduction, exactly as in the RY Cnc notebook: `apply_standard_rejections()`, keep
non-rejected detections. Then two versions of the same fit — ordinary least squares on every
detection, and on **5-year median bins** (closer to Schaefer's method, and robust to the wild
variation in how many plates each era contributed). Slope is quoted in mag/century,
**positive = getting fainter**.

In [ ]:
def cleaned_detections(local_id):
    """Our one standard reduction, applied identically to target and controls."""
    lc = sess.lightcurve(local_id)
    lc.apply_standard_rejections()
    det = lc.keep_only.nonrej_detected()
    return det["time"].jyear, np.asarray(det["magcal_magdep"], dtype=float)


def slope_mag_per_century(year, mag):
    """OLS slope of mag vs. time, in mag per century (positive = dimming)."""
    design = np.vstack([np.ones_like(year), (year - 1940) / 100]).T
    coef, *_ = np.linalg.lstsq(design, mag, rcond=None)
    return coef[0], coef[1]           # intercept at 1940, slope


def median_bins(year, mag, width=5, min_points=5):
    edges = np.arange(1885, 1996, width)
    centers, medians = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        sel = (year >= lo) & (year < hi)
        if sel.sum() >= min_points:
            centers.append((lo + hi) / 2)
            medians.append(np.median(mag[sel]))
    return np.asarray(centers), np.asarray(medians)


year, mag = cleaned_detections(0)
bin_year, bin_mag = median_bins(year, mag)
_, slope_all = slope_mag_per_century(year, mag)
b0, slope_binned = slope_mag_per_century(bin_year, bin_mag)

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.scatter(year, mag, s=4, color="lightsteelblue", label=f"{len(year)} cleaned detections")
ax.plot(bin_year, bin_mag, "o-", color="#39516b", label="5-year median bins")
grid = np.array([1889, 1990])
ax.plot(grid, b0 + slope_binned * (grid - 1940) / 100, "--", color="crimson",
        label=f"OLS on bins: {slope_binned:+.3f} mag/century")
ax.axvspan(1953, 1969, color="orange", alpha=0.15)
ax.invert_yaxis()
ax.set_xlabel("Year")
ax.set_ylabel("Photographic magnitude")
ax.set_title("KIC 8462852 in DASCH DR7, standard reduction")
ax.legend(loc="upper left")
plt.show()

print(f"slope, every detection : {slope_all:+.3f} mag/century")
print(f"slope, 5-yr median bins: {slope_binned:+.3f} mag/century")
print(f"Schaefer (2016)        : +0.164 \u00b1 0.013 mag/century")

**Reproduced.** An independent pipeline (DR7 photometry, daschlab standard rejections, no
per-plate hand-vetting) lands within a few thousandths of Schaefer's headline number. Whatever
this slope *is*, it is a property of the data, not an arithmetic slip — reading the fit line
off the plot, the star's binned magnitudes really do sit ~0.15 mag fainter in the 1980s than
in the 1920s. Score one for Schaefer.

## 3. Measurement 2: trend, or step?

Now Hippke's structural objection. A straight line fitted across a 16-year hole will happily
convert a *calibration offset* between the two sides into a *slope*. The test costs three
lines: fit **only the pre-gap data** (1889–1953, which is 75% of the detections), extrapolate
that fit across the gap, and ask how far the post-gap data sits from the extrapolation.

In [ ]:
pre = year <= 1953
post = year >= 1969

pre_b0, pre_slope = slope_mag_per_century(year[pre], mag[pre])
post_median = np.median(mag[post])
post_epoch = np.median(year[post])
extrapolated = pre_b0 + pre_slope * (post_epoch - 1940) / 100
step = post_median - extrapolated

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(bin_year[bin_year <= 1953], bin_mag[bin_year <= 1953], "o-", color="#39516b",
        label="5-yr bins, pre-gap")
ax.plot(bin_year[bin_year >= 1969], bin_mag[bin_year >= 1969], "s-", color="firebrick",
        label="5-yr bins, post-gap")
grid = np.array([1889, 1990])
ax.plot(grid, pre_b0 + pre_slope * (grid - 1940) / 100, "--", color="#39516b", alpha=0.7,
        label=f"pre-gap fit: {pre_slope:+.3f} mag/century, extrapolated")
ax.axvspan(1953, 1969, color="orange", alpha=0.15)
ax.invert_yaxis()
ax.set_xlabel("Year")
ax.set_ylabel("Photographic magnitude")
ax.set_title("The same data, split at the Menzel Gap")
ax.legend(loc="upper left")
plt.show()

print(f"pre-gap detections : {pre.sum()}   post-gap: {post.sum()}")
print(f"pre-gap slope alone: {pre_slope:+.3f} mag/century")
print(f"post-gap median is {step:+.3f} mag from the pre-gap extrapolation")

Score one for Hippke. Sixty-four years of pre-gap plates, on their own, are **nearly flat**
(~+0.02 mag/century). Essentially the entire fitted "dimming" lives in a ~0.09 mag *offset*
between the pre-gap and post-gap eras — precisely where the observatory changed telescopes,
emulsions, and observing programmes. That is fully consistent with a calibration step... and
*also* fully consistent with a star that genuinely faded during the sixteen years nobody in
Cambridge was looking. The target's own light curve cannot distinguish these. Only other
stars can.

## 4. Measurement 3: the differential test

If the step is instrumental, it should afflict other stars on the same plates. So: take field
stars from the same reference catalog and run them through the *character-for-character same
pipeline*, then see where our star falls in the resulting slope distribution.

**Selection, written down before running** (per the [explorations rules](README.md)):

- `class == 0` (catalog point source), not the target itself;
- brightness within **±1.0 mag** of the target's B = 12.36, so plate limits bite similarly;
- **color within ±0.25** of the target's +0.51 — this matters: a first color-blind pass pulled
  in a red giant and an AGB-candidate, and evolved red stars both *genuinely* vary on long
  timescales and stress the blue-plate calibration (it is why Lund et al. compared against
  F stars specifically);
- at least 800 cleaned detections;
- the 14 closest in brightness that pass.

In [ ]:
stdmag = np.asarray(refcat["stdmag"], dtype=float)
color = np.asarray(refcat["color"], dtype=float)
local_ids = np.asarray(refcat["local_id"])

eligible = (
    np.isfinite(stdmag) & np.isfinite(color)
    & (np.asarray(refcat["class"]) == 0) & (local_ids != 0)
    & (np.abs(stdmag - float(target["stdmag"])) < 1.0)
    & (np.abs(color - float(target["color"])) < 0.25)
)
candidates = np.flatnonzero(eligible)
candidates = candidates[np.argsort(np.abs(stdmag[candidates] - float(target["stdmag"])))][:14]

controls = []
for idx in candidates:
    lid = int(local_ids[idx])
    c_year, c_mag = cleaned_detections(lid)
    if len(c_year) < 800:
        continue
    cb_year, cb_mag = median_bins(c_year, c_mag)
    _, c_slope = slope_mag_per_century(cb_year, cb_mag)
    controls.append((lid, stdmag[idx], color[idx], len(c_year), c_slope))
    print(f"control {lid:4d}: B {stdmag[idx]:5.2f}  color {color[idx]:+.2f}  "
          f"{len(c_year):4d} detections  slope {c_slope:+.3f} mag/century")

ctrl_slopes = np.array([c[4] for c in controls])
print(f"\n{len(controls)} controls: median {np.median(ctrl_slopes):+.3f}, "
      f"scatter (std) {ctrl_slopes.std():.3f} mag/century")
print(f"target                : {slope_binned:+.3f} mag/century")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
rng = np.random.default_rng(11)
ax.scatter(ctrl_slopes, rng.uniform(-0.25, 0.25, size=ctrl_slopes.size),
           s=45, color="#39516b", alpha=0.8, label="controls (matched B & color)")
ax.axvline(slope_binned, color="crimson", lw=2, label=f"KIC 8462852: {slope_binned:+.3f}")
ax.axvline(0, color="gray", lw=0.8)
ax.set_ylim(-1, 1)
ax.set_yticks([])
ax.set_xlabel("Fitted slope (mag/century, positive = dimming)")
ax.set_title("The same pipeline, applied to 14 matched field stars")
ax.legend(loc="upper left")
plt.show()

extreme = [c for c in controls if abs(c[4]) > 0.25]
print(f"controls with |slope| > 0.25 mag/century: {[c[0] for c in extreme]}")
print(f"controls more *positive* than the target: "
      f"{(ctrl_slopes > slope_binned).sum()} of {len(ctrl_slopes)}")

What are those extreme controls — pipeline victims, or genuinely variable stars we failed to
exclude? A SIMBAD cone search on each (best-effort; skipped gracefully if offline):

In [ ]:
try:
    from astropy import units as u
    from astroquery.simbad import Simbad

    simbad = Simbad()
    simbad.add_votable_fields("otype")
    for lid, b, col, n, s in extreme:
        row = refcat[np.flatnonzero(local_ids == lid)[0]]
        res = simbad.query_region(row["pos"], radius=8 * u.arcsec)
        found = f"{res[0]['main_id']} (otype {res[0]['otype']})" if res is not None and len(res) else "no SIMBAD entry"
        print(f"control {lid} (slope {s:+.3f}): {found}")
except Exception as err:   # SIMBAD is a live service; don't let it break the notebook
    print(f"SIMBAD unavailable ({err}); outliers left uninvestigated this run")

## 5. The verdict we can defend

Putting the three measurements side by side:

| measurement | result | favors |
|---|---|---|
| full-series slope, standard reduction | **+0.17 mag/century** — Schaefer's 0.164 ± 0.013, reproduced | Schaefer |
| pre-gap (1889–1953) slope alone | **+0.02 mag/century** — flat; the signal is a **+0.09 mag step** across the gap | Hippke/Lund |
| matched-control slope scatter | **~0.14 mag/century** std; sun-colored field stars reach \|0.36\| under the identical pipeline | Hippke/Lund |

**Our opinion, and why.** Schaefer measured the data correctly — we get his number with an
independent pipeline. But his ±0.013 is a *statistical* error, and measurement 3 says the
*systematic* floor for this kind of naive century slope is an order of magnitude larger:
ordinary stars of the same brightness and color, on the same plates, routinely "trend" at
tenths of a magnitude per century. With the empirical error bar, the claim reads
**+0.17 ± 0.14 mag/century** — real as a feature of the data, unconvincing as astrophysics.
Add measurement 2 — the signal is not a steady slide but a step across the exact years the
observatory reorganized — and the Hippke/Lund reading is the better-supported one *from DASCH
data alone*. The star is still the most-dimming object of the 15 we fit, and Montet & Simon
prove it dims in the modern era, so Schaefer could well be *right* — but these plates, reduced
this way, cannot demonstrate it. That is presumably why the dispute is a decade old.

**What would move the needle** (honest caveats, and the next layer of work):

- Our reduction is deliberately generic. Schaefer hand-vetted plates and used local comparison
  ensembles; a serious replication would too — daschlab's per-point plate **cutouts**
  (notebook 1) are exactly the tool for adjudicating his rejected-vs-kept plates.
- The extreme controls deserve forensics: unknown variables, blends, or proper-motion
  neighbors drifting through the aperture would each inflate our empirical error bar unfairly.
- The decisive test in the literature was never the slope but the **gap-step ensemble**: does
  the target's step exceed the *distribution of steps* in a large control sample? That is a
  natural, well-scoped follow-up — and this notebook's `cleaned_detections` + `median_bins`
  machinery already does everything it needs.

*Runtime note: first run downloads ~15 light curves (a few minutes); everything caches under
`data/cache/dasch/kic8462852/` (~25 MB), so re-runs are fast and offline apart from the
optional SIMBAD lookup.*